# Phase 1 export: split files and MC Dropout logits (existing checkpoint)

**Decision-support prototype only. Not a diagnosis. Consult a qualified clinician.**

This small notebook does NOT train anything. It (1) rebuilds your original HAM10000 split and checks it matches the
original run, (2) saves the split as CSV files, and (3) saves the raw logits of all 10 MC Dropout passes for the
validation and test sets, so temperature scaling and threshold choice can be done later on a laptop CPU.
Run the 3 code cells in order. Runtime: GPU is optional (T4 takes about 2 minutes, CPU about 15).

## Cell 1 of 3: setup, download HAM10000, rebuild and verify the split
Uses the same code and seed as the original training notebook (cell 12). The cell stops with an error if the split
does not match the original counts, so we never export logits for a different split by accident.
You will be asked to upload two files: `kaggle.json` (never commit it) and `efficientnet_b3_best.pth`.

In [ ]:
!pip -q install scikit-learn
import os, glob, json, random
import numpy as np, pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold
from google.colab import files

# ---------------- ONE CONFIG BLOCK (all seeds / numbers live here) ----------------
CFG = dict(
    split_seed=42,            # same as the original notebook: DO NOT CHANGE
    n_folds=7,                # fold 0 = test, fold 1 = val, folds 2-6 = train (original recipe)
    mc_seed=1234,             # seed for the dropout masks of this one-off export
    mc_passes=10,             # must equal mc_dropout_passes in model_config.json
    input_size=300,
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    class_codes=["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"],
    # Expected class counts from docs/results/split_class_counts.csv (the original run)
    expected={"train": dict(akiec=230, bcc=358, bkl=772, df=89, mel=782, nv=4796, vasc=89),
              "val":   dict(akiec=63,  bcc=67,  bkl=159, df=17, mel=185, nv=935,  vasc=32),
              "test":  dict(akiec=34,  bcc=89,  bkl=168, df=9,  mel=146, nv=974,  vasc=21)},
    expected_test_total=1441,
)
OUT = "/content/phase1_out"; os.makedirs(OUT, exist_ok=True)
random.seed(CFG["split_seed"]); np.random.seed(CFG["split_seed"]); torch.manual_seed(CFG["split_seed"])

# kaggle.json -> stored only in this Colab session
if not os.path.exists("/root/.kaggle/kaggle.json"):
    up = files.upload()   # choose kaggle.json
    os.makedirs("/root/.kaggle", exist_ok=True)
    open("/root/.kaggle/kaggle.json", "wb").write(up["kaggle.json"]); os.chmod("/root/.kaggle/kaggle.json", 0o600)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham --unzip -q

meta = pd.read_csv("/content/ham/HAM10000_metadata.csv")
img_index = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob("/content/ham/**/*.jpg", recursive=True)}
meta["path"] = meta["image_id"].map(img_index)
assert meta["path"].notna().all(), "some images are missing"
meta["label"] = meta["dx"].map({c: i for i, c in enumerate(CFG["class_codes"])})

# Same split recipe as the original notebook
sgkf = StratifiedGroupKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["split_seed"])
folds = list(sgkf.split(meta, meta["label"], groups=meta["lesion_id"]))
idx = dict(test=folds[0][1], val=folds[1][1], train=np.concatenate([folds[i][1] for i in range(2, 7)]))
splits = {k: meta.iloc[v] for k, v in idx.items()}

# ---- verify against the original run ----
for name, df in splits.items():
    got = df["dx"].value_counts().to_dict()
    assert got == CFG["expected"][name], f"{name} split differs from original!\n got={got}\n want={CFG['expected'][name]}"
assert len(splits["test"]) == CFG["expected_test_total"]
assert not set(splits["train"].lesion_id) & set(splits["val"].lesion_id)
assert not set(splits["train"].lesion_id) & set(splits["test"].lesion_id)
assert not set(splits["val"].lesion_id) & set(splits["test"].lesion_id)
print("OK: split matches the original run. sizes:", {k: len(v) for k, v in splits.items()})

for name, df in splits.items():
    df[["image_id", "lesion_id", "dx"]].to_csv(f"{OUT}/split_{name}.csv", index=False)
print(os.listdir(OUT))

## Cell 2 of 3: load the checkpoint and sanity-check it
Upload `backend/model/efficientnet_b3_best.pth` when asked. We rebuild the model exactly as the backend does and
check that plain (non-MC) test accuracy is about 83.5%, the number in `model_config.json`. If it is not, the checkpoint or split is wrong and we stop.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not os.path.exists("/content/efficientnet_b3_best.pth"):
    up = files.upload()   # choose efficientnet_b3_best.pth
    open("/content/efficientnet_b3_best.pth", "wb").write(up["efficientnet_b3_best.pth"])

model = models.efficientnet_b3(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CFG["class_codes"]))
model.load_state_dict(torch.load("/content/efficientnet_b3_best.pth", map_location=device))
model = model.to(device).eval()

# Resize straight to 300x300 (no crop) + ImageNet normalisation = the backend contract
eval_tf = transforms.Compose([transforms.Resize((CFG["input_size"], CFG["input_size"])),
                              transforms.ToTensor(), transforms.Normalize(CFG["mean"], CFG["std"])])

class ImgSet(Dataset):
    def __init__(self, df): self.paths, self.labels = df["path"].values, df["label"].values
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return eval_tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

def loader(df): return DataLoader(ImgSet(df), batch_size=32, shuffle=False, num_workers=2)   # order must stay fixed

# plain eval accuracy on test (no dropout) = sanity check
correct = 0
with torch.no_grad():
    for x, y in loader(splits["test"]):
        correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
acc = correct / len(splits["test"])
print(f"plain test accuracy = {acc:.4f}  (config says 0.8348)")
assert abs(acc - 0.8348) < 0.01, "checkpoint/split mismatch: stop and tell Claude"

## Cell 3 of 3: export MC Dropout logits and download
Backbone runs once per batch; the classifier runs 10 times with dropout on. We keep the raw **logits** of every pass
(not softmax), so temperature can later be applied to each pass BEFORE softmax and averaging. No autocast, same as the backend.

In [ ]:
def mc_logits(df, seed: int) -> np.ndarray:
    """Return logits of shape (n_images, passes, 7). Dropout masks are seeded for repeatability."""
    torch.manual_seed(seed)
    out = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader(df):
            feats = F.adaptive_avg_pool2d(model.features(x.to(device)), 1).flatten(1)   # backbone once
            model.classifier.train()                                                    # dropout ON (classifier only)
            out.append(torch.stack([model.classifier(feats) for _ in range(CFG["mc_passes"])], 1).cpu())
            model.classifier.eval()
    return torch.cat(out).numpy().astype(np.float32)

for name in ["val", "test"]:
    df = splits[name]
    L = mc_logits(df, CFG["mc_seed"])
    np.savez_compressed(f"{OUT}/mc_logits_{name}.npz", logits=L, labels=df["label"].values.astype(np.int64),
                        image_ids=df["image_id"].values.astype(str), lesion_ids=df["lesion_id"].values.astype(str),
                        class_codes=np.array(CFG["class_codes"]), seed=CFG["mc_seed"], passes=CFG["mc_passes"])
    print(name, L.shape)

json.dump({k: CFG[k] for k in ["split_seed", "mc_seed", "mc_passes"]}, open(f"{OUT}/export_info.json", "w"))
!cd /content/phase1_out && zip -q -r /content/phase1_out.zip .
files.download("/content/phase1_out.zip")